<a href="https://colab.research.google.com/github/MKpng/PPD/blob/main/ppd_phase_one.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Steam Games - Phase 1 Analysis
## Dataset: [CSV Files](https://drive.google.com/drive/folders/1t-Zyudaot0C7BOxQs79PtcENZgEpnNEi?usp=sharing)
---
## Context
>This notebook answers three business questions about the Steam platform using
raw CSV processing with Python's built-in `csv` module.<br>
>The chellenge is to answer the questions without using `Pandas`, `Numpy`, `Matplotlib` or other libraries.
---
## Business Questions

**Question 1:** What is the percentage of free and paid games on the platform?

**Question 2:** Which year had the highest number of new game releases?

**Question 3:** What is the total number of players across free and paid games?

## Requirement Imports

In [ ]:
import csv # ables csv file manipulation.
import random # handles randomization.
from collections import defaultdict # handles missing keys with a default value (no need for try / error overdo).

## Generating a sample dataset (testing purposes) function

In [ ]:
def get_sample(size=20):
    """
    Generates a random sample CSV from the Steam games dataset.

    Reads the full dataset from 'steam_games.csv', selects a random
    sample of rows without replacement, and writes the result to
    'sample_games.csv' preserving the original header.

    Parameters
    ----------
    size : int, optional
        Number of rows to sample. Defaults to 20.

    Returns
    -------
    None

    Raises
    ------
    FileNotFoundError
        If 'steam_games.csv' is not found.
    ValueError
        If size is larger than the number of rows in the dataset.

    Examples
    --------
    >>> get_sample(100)
    """
    my_dict = []
    with open('steam_games.csv', newline='', encoding='utf-8') as f:
        r = csv.reader(f)
        header = next(r)
        for i in r:
            my_dict.append(i)

    rand_pos = random.sample(range(len(my_dict)), size)

    with open('sample_games.csv', 'w', newline='', encoding='utf-8') as w:
        writer = csv.writer(w)
        writer.writerow(header)
        for i in rand_pos:
            writer.writerow(my_dict[i])

## Dictionary function

In [ ]:
# Open file and ensuring encoding to UTF-8 for no Unicode errors.
def get_dict(file):
    """
    Builds a summary dictionary from the Steam games CSV dataset.

    Reads the CSV file in a single pass and aggregates key metrics
    including game counts by year, free vs paid distribution, and
    player counts. Uses defaultdict internally for performance.

    Parameters
    ----------
    file : str
        Path to the CSV file to be read. Expected to be the
        Steam games dataset or a Sample dataset generated by get_sample().

    Returns
    -------
    dict
        A dictionary containing the following keys:

        - ``years`` (dict): Game count per release year.
        - ``free`` (int): Total number of free games.
        - ``paid`` (int): Total number of paid games.
        - ``free_players`` (int): Total players across free games.
        - ``paid_players`` (int): Total players across paid games.
        - ``player_count`` (int): Total players across all games.
        - ``game_count`` (int): Total number of games.
        - ``max_games`` (int): Most games released in a year within all years.
        - ``year_games`` (list): Year(s) with most games released.

    Raises
    ------
    FileNotFoundError
        If the file specified by `file` does not exist.
    ValueError
        If player count values cannot be cast to int.

    Examples
    --------
    >>> data = get_dict('steam_games.csv')
    >>> data = get_dict('sample_games.csv')
    """
    data = defaultdict(int)
    data['years'] = defaultdict(int)

    with open(file, newline='', encoding='utf-8') as f:
        reader = csv.reader(f)
        next(reader)

        for i in reader:
            data['years'][i[2][-4:]] += 1
            if i[6] == "0.0":
                data['free'] += 1
                data['free_players'] += int(i[4])
            else:
                data['paid'] += 1
                data['paid_players'] += int(i[4])

    data['player_count'] = data['paid_players'] + data['free_players']
    data['game_count'] = data['paid'] + data['free']
    data['max_games'] = max(data['years'].values())
    data['year_games'] = [k for k, v in data['years'].items() if v == data['max_games']]

    return dict(data)

## Initializing Dictionary

### Working with the whole dataset (`steam_games.csv`)

In [ ]:
data = get_dict('steam_games.csv')

### Working with the Sample Dataset (`sample_games.csv`)

> `sample_games.csv` is already provided alongside the main dataset.  
> Uncommenting `get_sample()` will overwrite `sample_games.csv` with newly sampled data.  
> To use the sample dataset, uncomment one or both lines below and comment out `data = get_dict('steam_games.csv')`.  
> To use the whole dataset, keep the two lines below commented out.

In [ ]:
#get_sample()
#data = get_dict('sample_games.csv')

## Question 1 — Free vs Paid Games
> *What is the percentage of free and paid games on the platform?*

In [ ]:
# Question 1
total = data['game_count']
print(f"Total games: {total} [100%]")
print(f"Paid games:  {data['paid']} [{data['paid'] * 100 / total:.2f}%]")
print(f"Free games:  {data['free']} [{data['free'] * 100 / total:.2f}%]")

Total games: 72934 [100%]
Paid games:  60254 [82.61%]
Free games:  12680 [17.39%]


## Question 2 — Most Active Release Year
> *Which year had the highest number of new game releases?*

In [ ]:
# Question 2
print(f"The year(s) with the most releases: {', '.join(data['year_games'])} with {data['max_games']} games!")

The year(s) with the most releases: 2022 with 13961 games!


## Question 3 — Player Distribution
> *What is the total number of players across free and paid games?*

In [ ]:
# Question 3
print(f"The maximum number of simultaneous users across all games in this dataset is {data['player_count']} players.\n")
print(f"Out of all these players:\n")
print(f"  {data['paid_players']} [{data['paid_players'] * 100 / data['player_count']:.2f}%] play paid games.")
print(f"  {data['free_players']} [{data['free_players'] * 100 / data['player_count']:.2f}%] play free games.")

The maximum number of simultaneous users across all games in this dataset is 10218294 players.

Out of all these players:

  6861567 [67.15%] play paid games.
  3356727 [32.85%] play free games.
